In [0]:
display(dbutils.fs.ls("/Workspace/Users/"))

In [0]:
display(
    dbutils.fs.ls(
        "/Workspace/Users/volkan.kulturoglu@windowslive.com/"
    )
)

In [0]:
display(
    dbutils.fs.ls("/Workspace/Users/volkan.kulturoglu@windowslive.com/")
)

In [0]:
display(
    dbutils.fs.ls(
        "/Workspace/Users/volkan.kulturoglu@windowslive.com/DbAcademy/"
    )
)

In [0]:
base_path = "/Workspace/Users/volkan.kulturoglu@windowslive.com/DbAcademy/Data/"

In [0]:
display(dbutils.fs.ls(base_path))

In [0]:
product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "product.csv")
)

display(product_df)

In [0]:
print("Rows:", product_df.count())
print("Columns:", len(product_df.columns))

product_df.printSchema()

In [0]:
product_df.select(
    "PRODUCT_ID",
    "MANUFACTURER",
    "DEPARTMENT",
    "BRAND",
    "COMMODITY_DESC",
    "SUB_COMMODITY_DESC"
).display()

In [0]:
transaction_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "transaction_data.csv")
)

product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "product.csv")
)

demographic_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "hh_demographic.csv")
)

campaign_desc_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "campaign_desc.csv")
)

campaign_table_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "campaign_table.csv")
)

coupon_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "coupon.csv")
)

coupon_redempt_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "coupon_redempt.csv")
)

In [0]:
transaction_df

In [0]:
datasets = {
    "transactions": transaction_df,
    "products": product_df,
    "demographics": demographic_df,
    "campaign_desc": campaign_desc_df,
    "campaign_table": campaign_table_df,
    "coupons": coupon_df,
    "coupon_redemptions": coupon_redempt_df
}

for name, df in datasets.items():
    print(f"{name}: {df.count():,} rows x {len(df.columns)} columns")

In [0]:
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    df.printSchema()

In [0]:
from pyspark.sql import functions as F

product_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("PRODUCT_ID").alias("unique_product_ids")
).display()

demographic_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("household_key").alias("unique_households")
).display()

campaign_desc_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("CAMPAIGN").alias("unique_campaigns")
).display()

In [0]:
campaign_table_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("CAMPAIGN", "household_key").alias("unique_campaign_households")
).display()

In [0]:
missing_products = (
    transaction_df
    .select("PRODUCT_ID")
    .distinct()
    .join(
        product_df.select("PRODUCT_ID"),
        on="PRODUCT_ID",
        how="left_anti"
    )
)

missing_products.count()

In [0]:
def check_referential_integrity(
    source_df,
    source_key,
    reference_df,
    reference_key
):
    unmatched = (
        source_df
        .select(source_key)
        .distinct()
        .join(
            reference_df.select(reference_key).distinct(),
            source_df[source_key] == reference_df[reference_key],
            "left_anti"
        )
    )
    
    return unmatched.count()

In [0]:
print(
    "Campaign Table → Campaign Desc:",
    check_referential_integrity(
        campaign_table_df, "CAMPAIGN",
        campaign_desc_df, "CAMPAIGN"
    )
)

print(
    "Coupons → Products:",
    check_referential_integrity(
        coupon_df, "PRODUCT_ID",
        product_df, "PRODUCT_ID"
    )
)

print(
    "Coupons → Campaign Desc:",
    check_referential_integrity(
        coupon_df, "CAMPAIGN",
        campaign_desc_df, "CAMPAIGN"
    )
)

print(
    "Coupon Redemptions → Campaign Desc:",
    check_referential_integrity(
        coupon_redempt_df, "CAMPAIGN",
        campaign_desc_df, "CAMPAIGN"
    )
)

In [0]:
transaction_households = (
    transaction_df
    .select("household_key")
    .distinct()
    .count()
)

demographic_households = (
    demographic_df
    .select("household_key")
    .distinct()
    .count()
)

matched_households = (
    transaction_df
    .select("household_key")
    .distinct()
    .join(
        demographic_df.select("household_key").distinct(),
        on="household_key",
        how="inner"
    )
    .count()
)

print("Transactional households:", transaction_households)
print("Demographic households:", demographic_households)
print("Matched households:", matched_households)

In [0]:
coverage = matched_households / transaction_households * 100

print(f"Demographic coverage: {coverage:.2f}%")

In [0]:
demographics_without_transactions = (
    demographic_df
    .select("household_key")
    .distinct()
    .join(
        transaction_df.select("household_key").distinct(),
        on="household_key",
        how="left_anti"
    )
    .count()
)

print(
    "Demographic households without transactions:",
    demographics_without_transactions
)

In [0]:
from pyspark.sql import functions as F

def null_profile(df):
    return df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

In [0]:
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    display(null_profile(df))

In [0]:
transaction_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("BASKET_ID", "PRODUCT_ID").alias("unique_basket_products")
).display()

In [0]:
coupon_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(
        "COUPON_UPC",
        "PRODUCT_ID",
        "CAMPAIGN"
    ).alias("unique_combinations")
).display()

In [0]:
coupon_duplicates = (
    coupon_df
    .groupBy("COUPON_UPC", "PRODUCT_ID", "CAMPAIGN")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(coupon_duplicates)

In [0]:
print("Duplicate combinations:", coupon_duplicates.count())

coupon_duplicates.groupBy("count").count().orderBy("count").display()

In [0]:
example = (
    coupon_duplicates
    .filter(F.col("count") == 19)
    .first()
)

display(
    coupon_df.filter(
        (F.col("COUPON_UPC") == example["COUPON_UPC"]) &
        (F.col("PRODUCT_ID") == example["PRODUCT_ID"]) &
        (F.col("CAMPAIGN") == example["CAMPAIGN"])
    )
)

In [0]:
coupon_redempt_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(
        "household_key",
        "DAY",
        "COUPON_UPC",
        "CAMPAIGN"
    ).alias("unique_redemptions")
).display()

In [0]:
transaction_df.select(
    F.min("DAY").alias("min_day"),
    F.max("DAY").alias("max_day"),
    F.min("WEEK_NO").alias("min_week"),
    F.max("WEEK_NO").alias("max_week"),
    F.min("TRANS_TIME").alias("min_time"),
    F.max("TRANS_TIME").alias("max_time"),
    F.min("QUANTITY").alias("min_quantity"),
    F.max("QUANTITY").alias("max_quantity"),
    F.min("SALES_VALUE").alias("min_sales"),
    F.max("SALES_VALUE").alias("max_sales"),
    F.min("RETAIL_DISC").alias("min_retail_disc"),
    F.max("RETAIL_DISC").alias("max_retail_disc"),
    F.min("COUPON_DISC").alias("min_coupon_disc"),
    F.max("COUPON_DISC").alias("max_coupon_disc")
).display()

In [0]:
transaction_df.select(
    F.expr("percentile_approx(QUANTITY, array(0.5, 0.9, 0.95, 0.99, 0.999))").alias("quantity_percentiles"),
    F.expr("percentile_approx(SALES_VALUE, array(0.5, 0.9, 0.95, 0.99, 0.999))").alias("sales_percentiles")
).display()

In [0]:
high_quantity_products = (
    transaction_df
    .filter(F.col("QUANTITY") > 100)
    .join(
        product_df.select(
            "PRODUCT_ID",
            "DEPARTMENT",
            "COMMODITY_DESC",
            "SUB_COMMODITY_DESC",
            "CURR_SIZE_OF_PRODUCT"
        ),
        on="PRODUCT_ID",
        how="left"
    )
    .select(
        "household_key",
        "BASKET_ID",
        "PRODUCT_ID",
        "QUANTITY",
        "SALES_VALUE",
        "DEPARTMENT",
        "COMMODITY_DESC",
        "SUB_COMMODITY_DESC",
        "CURR_SIZE_OF_PRODUCT"
    )
    .orderBy(F.desc("QUANTITY"))
)

display(high_quantity_products.limit(30))

In [0]:
transaction_df.select(
    F.count("*").alias("total_transactions"),
    F.sum((F.col("QUANTITY") > 100).cast("int")).alias("quantity_over_100"),
    F.sum((F.col("QUANTITY") > 1000).cast("int")).alias("quantity_over_1000"),
    F.sum((F.col("QUANTITY") > 10000).cast("int")).alias("quantity_over_10000")
).display()

In [0]:
campaign_desc_df.select(
    F.min("START_DAY").alias("min_start_day"),
    F.max("START_DAY").alias("max_start_day"),
    F.min("END_DAY").alias("min_end_day"),
    F.max("END_DAY").alias("max_end_day")
).display()

In [0]:
campaign_desc_df.filter(
    F.col("START_DAY") > F.col("END_DAY")
).display()

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)